In [1]:
# import necessary libraries
import os
import torch 
import torch.nn as nn
from spnc import spnc_anisotropy
import numpy as np
import matplotlib.pyplot as plt
import tqdm as tqdm
import pickle
import spnc_ml as ml


from pathlib import Path


CANDIDATES = [
    
    Path(r"C:\Users\tom\Desktop\Repository"),
    Path(r"C:\Users\Chen\Desktop\Repository"),
    Path(r"/Users/vvvp./Desktop"),
]
searchpaths = [p for p in CANDIDATES if p.exists()]

#tuple of repos
repos = ('machine_learning_library',)

from deterministic_mask import fixed_seed_mask, max_sequences_mask
import repo_tools
repo_tools.repos_path_finder(searchpaths, repos)
from single_node_res import single_node_reservoir
import ridge_regression as RR
from linear_layer import *
from mask import binary_mask
from utility import *
from NARMA10 import NARMA10
from datasets.load_TI46_digits import *
import datasets.load_TI46 as TI46
from sklearn.metrics import classification_report

from spnc_ml import spnc_TI46, spnc_narma10


In [2]:
# 构建储层对象
class ReservoirParams:
    def __init__(self, **kwargs):
            # Reservoir parameters 
            self.h = 0.4
            self.theta_H = 90
            self.k_s_0 = 0
            self.phi = 45
            self.beta_prime = 35.13826524755751

            # Network parameters 
            self.Nvirt = 50
            self.m0 = 0.005288612874870094
            self.bias = True
            self.Nwarmup = 0
            self.verbose_repr = False

            self.params = {
                'theta': 0.34142235979698393,
                'gamma': 0.069274461903986,
                'delay_feedback': 0,
                'Nvirt': self.Nvirt,
                'length_warmup': self.Nwarmup,
                'warmup_sample': self.Nwarmup * self.Nvirt,
                'voltage_noise': False,
                'seed_voltage_noise': 1234,
                'delta_V': 0.1,
                'johnson_noise': False,
                'seed_johnson_noise': 1234,
                'mean_johnson_noise': 0.0000,
                'std_johnson_noise': 0.00001,
                'thermal_noise': False,
                'seed_thermal_noise': 1234,
                'lambda_ou': 1.0,
                'sigma_ou': 0.1
        }

            for key in ['h', 'theta_H', 'k_s_0', 'phi', 'beta_prime', 'Nvirt', 'm0', 'bias', 'Nwarmup']:
                if key in kwargs:
                    setattr(self, key, kwargs[key])

            
            if 'params' in kwargs and isinstance(kwargs['params'], dict):
                self.params.update(kwargs['params'])

    
    def update_params(self, **kwargs):
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, value)
            if key in self.params:
                self.params[key] = value
            if not hasattr(self, key) and key not in self.params:
                raise AttributeError(f"ReservoirParams has no attribute or param key '{key}'")
            
    def print_params(self, verbose=False):
        if not verbose:
            print(f"ReservoirParams(h={self.h}, beta_prime={self.beta_prime}, Nvirt={self.Nvirt})")
        else:
            print(f"ReservoirParams detailed info:")
            print(f"  h = {self.h}")
            print(f"  theta_H = {self.theta_H}")
            print(f"  k_s_0 = {self.k_s_0}")
            print(f"  phi = {self.phi}")
            print(f"  beta_prime = {self.beta_prime}")
            print(f"  Nvirt = {self.Nvirt}")
            print(f"  m0 = {self.m0}")
            print(f"  bias = {self.bias}")
            print("  params dictionary:")
            for k, v in self.params.items():
                print(f"    {k}: {v}")

In [3]:
def MSE(pred, desired):
    return np.mean(np.square(np.subtract(pred, desired)))

def NRMSE(pred, y_test, spacer=0.001):
    return np.sqrt(MSE(pred, y_test) / np.var(y_test))

In [4]:
def generate_signal(I,washout = 50,seed=1234):
    '''
    Generate a i.i.d. signal sequence with a [-1,1] range
    '''
    if seed is not None:
        np.random.seed(seed)
        
    signal_sequence = np.random.uniform(-1,1,size = I)

    washed_signal = signal_sequence[washout:]

    # Convert to 2D array
    washed_signal = washed_signal.reshape(-1,1)

    # print(np.shape(washed_signal))

    return washed_signal

def RidgeRegression(states, target, l, bias=True):
    # Ensure numpy
    if torch.is_tensor(states):
        states = states.detach().cpu().numpy()
    if torch.is_tensor(target):
        target = target.detach().cpu().numpy()
    if bias==True:
        # Add bias to states
        bias = np.ones((len(states), 1))
        states = np.concatenate((bias, states), axis=1)
    # Setup matrices from inputs
    M1 = np.matmul(states.transpose(), target) 
    M2 = np.matmul(states.transpose(), states)
    # Perform ridge regression
    weights = np.matmul(np.linalg.pinv(M2+l*np.identity(len(M2))), M1)
    return weights

def linear_MC(signal, states, splits=[0.2, 0.8], delays=50):
    # ensure flat input signal
    signal = np.asarray(signal).flatten()
    # generate target signal from delayed input signal
    shift = np.zeros((len(signal), delays))
    for i in range(len(signal)-delays):
        i += delays
        shift[i, :] = signal[i-delays:i]
    # split data
    wash, Ytrain, Ytest = np.split(shift, [int(len(signal)*splits[0]), int(len(signal)*splits[1])])
    wash, Xtrain, Xtest = np.split(states, [int(len(signal)*splits[0]), int(len(signal)*splits[1])])
    # sweep over range of hyperparameters gamma to find optimal MC
    bestMC = 0
    gammas = np.logspace(-10, 0, 11)
    for gamma in gammas:
        # Calculate weights
        weights = RidgeRegression(Xtrain, Ytrain, gamma, bias=False)
        # Predict test states
        prediction = np.matmul(Xtest, weights)
        # Loop over all delays k and evaluate MC_k
        MC_k = np.zeros(delays)
        for k in range(delays):
            # Take prediction and target for each delay
            pred = prediction[:, k]
            targ = Ytest[:, k]
            # Set up matrix to calculate covariance
            M = pred, targ
            # Calculate covariance
            coVarM = np.cov(M)
            # Take cov(xy) 
            coVar = coVarM[0,1]
            # Measure the variance of the signals
            outVar = np.var(pred)
            targVar = np.var(targ)
            # Calculate the total variance of the raw target and the specific
            # target
            totVar = outVar*targVar
            # If the covariance coefficient is greater than 0.1, treat as better
            # than random guessing and add to MC_k outputs
            if coVar**2/totVar > 0.1:
                MC_k[k] = coVar**2/totVar
        # Account for floating point errors in MC
        MC_k[MC_k>1] = 1
        # Sum memory capacity over all delays
        MC = sum(MC_k)
        # If best reported MC, save data
        if MC > bestMC:
            bestMC = MC
    return bestMC

def RunSpnc(signal,Nin,Nout,Nvirt,m0,transform, params,**kwargs):
    '''
    Run a reservoir computer with the signal sequence
    '''
    snr = single_node_reservoir(Nin, Nout, Nvirt, m0, res=transform)

    fixed_mask = kwargs.get('fixed_mask', False)
    if fixed_mask==True:
        # print("Deterministic mask will be used")
        seed_mask = kwargs.get('seed_mask', 1234)
        if seed_mask>=0:
            # print(seed_mask)
            snr.M = fixed_seed_mask(Nin, Nvirt, m0, seed=seed_mask)
        else:
            # print("Max_sequences mask will be used")
            snr.M = max_sequences_mask(Nin, Nvirt, m0)
    
    # Run
    S,_ = snr.transform(signal,params)
    
    return S
    
def evaluate_MC(reservoir_params, signal_len = 550, **kwargs):

    signal = generate_signal(signal_len, seed=kwargs.get('seed', 1234))
    # 打印signal的前10个元素


    spn = spnc_anisotropy(
        reservoir_params.h,
        reservoir_params.theta_H,
        reservoir_params.k_s_0,
        reservoir_params.phi,
        reservoir_params.beta_prime,
        restart=True
    )

    transform = spn.gen_signal_slow_delayed_feedback

    Output = RunSpnc(
        signal,
        1,                 
        1,       
        reservoir_params.Nvirt,
        reservoir_params.m0,
        transform,
        reservoir_params.params,
        fixed_mask=True,
        seed_mask=1234
    )


    MC = linear_MC(signal, Output, splits=[0.2,0.6], delays=10)

    return MC

In [5]:
def gen_KR_GR_input(Nreadouts, Nwash=10, seed=1234):
    # set seed
    np.random.seed(seed)
    # generate KR inputs
    KR_inputs = np.random.ranf((Nreadouts, Nwash))
    GR_inputs = np.tile(np.random.ranf((10)), (Nreadouts,1))
    all_inputs = np.concatenate((KR_inputs, GR_inputs), axis=1)
    # 打印all_inputs的前10个元素
    return all_inputs


def Evaluate_KR_GR(states, Nreadouts, threshold=0.1):
    GR_states = states[:,-1,:]
    '''
    Change the last 7 columns to GR states, the rest are KR states
    '''
    KR_states = states[:,-11,:]
    uGR, sGR, vGR = np.linalg.svd(GR_states)
    uKR, sKR, vKR = np.linalg.svd(KR_states)
    KR = 0
    GR = 0
    for i in range(Nreadouts):
        if sGR[i]>threshold:
            GR += 1
        if sKR[i]>threshold:
            KR += 1
    return KR, GR

# ------------------------ Reservoir ----------------------------


In [6]:
def evaluate_KRandGR(reservoir_params, Nreadouts=50, Nwash=10, **kwargs):
    
    Nreadouts= reservoir_params.Nvirt

    inputs = gen_KR_GR_input(Nreadouts, Nwash, seed=1234)   # <--- 用Nreadouts
    outputs = []
    for input_row in inputs:
        input_row = input_row.reshape(-1, 1)
        spn = spnc_anisotropy(reservoir_params.h, reservoir_params.theta_H,
                              reservoir_params.k_s_0, reservoir_params.phi,
                              reservoir_params.beta_prime, restart=True)
        transforms = spn.gen_signal_slow_delayed_feedback
        output = RunSpnc(input_row, 1, 1, reservoir_params.Nvirt,
                         reservoir_params.m0, transforms, reservoir_params.params, fixed_mask=True, seed_mask=1234)
        outputs.append(output)
    States = np.stack(outputs, axis=0)
    States = States/np.amax(States)
    KR, GR = Evaluate_KR_GR(States, Nreadouts, threshold=0.1)  # <--- 用Nreadouts
    return KR, GR


In [7]:
def eva_narma10(reservoir_params: ReservoirParams, 
                        Ntrain: int = 2000, Ntest: int = 1000):
        """评估NARMA-10任务"""

        # 创建储层
        spn = spnc_anisotropy(
            h=reservoir_params.h,
            theta_H=reservoir_params.theta_H,
            k_s=reservoir_params.k_s_0,
            phi=reservoir_params.phi,
            beta_prime=reservoir_params.beta_prime,
            restart=True
        )

        transform = spn.gen_signal_slow_delayed_feedback
        
        # 运行NARMA-10任务
        nrmse = spnc_narma10(
            Ntrain,
            Ntest,
            reservoir_params.Nvirt,
            reservoir_params.m0,
            reservoir_params.bias,
            transform,
            reservoir_params.params,
            seed_NARMA=1234,
            fixed_mask=True,
            seed_mask=1234,
            return_NRMSE=True,
        )
        
        # 计算NRMSE

        
        return nrmse

In [8]:
def eva_ti46(reservoir_params: ReservoirParams, 
                    speakers: ['f1','f2','f3','f4','f5']) -> float:
    """评估TI46任务"""
    
    # 创建储层
    spn = spnc_anisotropy(
        h=reservoir_params.h,
        theta_H=reservoir_params.theta_H,
        k_s=reservoir_params.k_s_0,
        phi=reservoir_params.phi,
        beta_prime=reservoir_params.beta_prime,
        restart=True
    )

    transform = spn.gen_signal_slow_delayed_feedback
    
    # 运行TI46任务
    accuracy = spnc_TI46(
        speakers,
        reservoir_params.Nvirt,
        reservoir_params.m0,
        reservoir_params.bias,
        transform,
        reservoir_params.params)
    
    return accuracy


### 以下cell是用来评估不同储层在threshold=0.1下的KR，GR数值

In [5]:
params_bestCQ = ReservoirParams(
        h=0.4, m0=0.006937322149792008, Nvirt=200, beta_prime=27.251620432439488,
        params={'theta': 0.01, 'gamma': 0.3663969812988086, 'Nvirt': 200}
    )

In [ ]:
results = evaluate_KRandGR(params_bestCQ)


In [9]:
print(results)

{'KR': 140, 'GR': 1}


In [5]:
params_bestMC = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	50.0,
        params={'theta': 0.1564938388583194, 'gamma': 0.04608425844940916, 'Nvirt': 200}
    )

In [6]:
results_bestMC = evaluate_KRandGR(params_bestMC)
print(results_bestMC)

{'KR': 6, 'GR': 3}


In [7]:
params_bestPhase = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	20.0,
        params={'theta': 0.07883177553412853, 'gamma': 0.09737503590304286, 'Nvirt': 200}
    )

In [8]:
results_bestPhase = evaluate_KRandGR(params_bestPhase)
print(results_bestPhase)

{'KR': 6, 'GR': 1}


In [9]:
params_109 = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	39.70001479261286,
        params={'theta': 0.14799158258968137, 'gamma': 0.05936342628024845, 'Nvirt': 200}
    )

In [10]:
results_109 = evaluate_KRandGR(params_109)
print(results_109)

{'KR': 6, 'GR': 3}


In [11]:
params_113 = ReservoirParams(
        h=0.4, m0=0.001, Nvirt=200, beta_prime=	20.0,
        params={'theta': 0.03442997633689481, 'gamma': 0.09508576109692872, 'Nvirt': 200})
results_113 = evaluate_KRandGR(params_113)
print(results_113)

{'KR': 45, 'GR': 1}


In [12]:
params_29 = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	50.0,
        params={'theta': 0.11577824609314408, 'gamma': 0.04447510407651761, 'Nvirt': 200})
results_29 = evaluate_KRandGR(params_29)
print(results_29)

{'KR': 6, 'GR': 3}


In [13]:
params_214 = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	20.0,
        params={'theta': 0.20890131642287738, 'gamma': 0.14586775378011477, 'Nvirt': 200}
    )
results_214 = evaluate_KRandGR(params_214)
print(results_214)

{'KR': 6, 'GR': 5}


In [14]:
params_138 = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	20.0,
        params={'theta': 0.6, 'gamma': 0.1172260326884624, 'Nvirt': 200}
    )
results_138 = evaluate_KRandGR(params_138)
print(results_138)

{'KR': 5, 'GR': 3}


In [15]:
params_326 = ReservoirParams(
        h=0.4, m0=0.07187768090200536, Nvirt=200, beta_prime= 28.21081994351713,
        params={'theta': 0.10419727159331738, 'gamma': 0.07187768090200536, 'Nvirt': 200}
    )
results_326 = evaluate_KRandGR(params_326)
print(results_326)

{'KR': 7, 'GR': 1}


### 以下cell用来评估delayed feedback=1情况下，MC和tasks的表现

In [9]:
speakers = ['f1','f2','f3','f4','f5'] 

In [10]:
# feedback =0的bestMC储层
params_bestMC = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	50.0,
        params={'theta': 0.1564938388583194, 'gamma': 0.04608425844940916, 'Nvirt': 200}
    )

#carry out MC, CQ, NARMA10, TI46
MC = evaluate_MC(params_bestMC)
KR, GR = evaluate_KRandGR(params_bestMC)
CQ = KR-GR
NRMSE = eva_narma10(params_bestMC)
ER = eva_ti46(params_bestMC,speakers)

print('bestMC without feedback: MC, CQ, KR,GR, NRMSE, ER',MC, CQ, KR,GR, NRMSE, ER)


Seed Training: 1234
len(x_train): 2000
error with zero =  0.004421615054149475
0.005753614601795441 0.6854114244875021
Samples for training:  500
Samples for test:  795
Using MFCC preprocessing
Nin = 13 , Nout =  10 , Nvirt =  200
Deterministic mask will be used
the shape of the mask is:  (200, 13)
Seed Training: 1234
len(x_train): 7999
error with zero =  0.06404390793420223
Optimal regression parameter =  0.007446583070924205
Train report
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        40
           1      0.952     1.000     0.976        40
           2      1.000     1.000     1.000        40
           3      1.000     1.000     1.000        40
           4      1.000     1.000     1.000        40
           5      0.976     1.000     0.988        40
           6      0.929     0.975     0.951        40
           7      0.974     0.925     0.949        40
           8      1.000     1.000     1.000        40
           9   

In [11]:
# feedback =0的bestMC储层
params_bestMC_df = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=200, beta_prime=	50.0,
        params={'theta': 0.1564938388583194, 'gamma': 0.04608425844940916, 'delay_feedback': 1, 'Nvirt': 200}
    )

#carry out MC, CQ, NARMA10, TI46
MC = evaluate_MC(params_bestMC_df)
KR, GR = evaluate_KRandGR(params_bestMC_df)
CQ = KR-GR
NRMSE = eva_narma10(params_bestMC_df)
ER = eva_ti46(params_bestMC_df,speakers)

print('bestMC with delayed_feedback: MC, CQ, KR, GR, NRMSE, ER',MC, CQ, KR, GR, NRMSE, ER)

Seed Training: 1234
len(x_train): 2000
error with zero =  0.0042254049028948165
0.005334360178951212 0.6599668523821256
Samples for training:  500
Samples for test:  795
Using MFCC preprocessing
Nin = 13 , Nout =  10 , Nvirt =  200
Deterministic mask will be used
the shape of the mask is:  (200, 13)
Seed Training: 1234
len(x_train): 7999
error with zero =  0.06785709093469663
Optimal regression parameter =  0.008229747049019877
Train report
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        40
           1      0.907     0.975     0.940        40
           2      1.000     1.000     1.000        40
           3      1.000     1.000     1.000        40
           4      1.000     1.000     1.000        40
           5      1.000     1.000     1.000        40
           6      0.907     0.975     0.940        40
           7      0.973     0.900     0.935        40
           8      1.000     1.000     1.000        40
           9  